# Notebook4 - ML Unsupervised for AD

使用 sliding-window features 與 Isolation Forest 進行多變量異常偵測，並用視覺化比較事件敏感度。

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def show_fig(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("..").resolve()

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
features = pd.read_csv(DATA_PROCESSED / "features.csv", parse_dates=["timestamp"])
event_catalog = pd.read_csv(DATA_SYNTHETIC / "synthetic_event_catalog.csv", parse_dates=["start_time", "end_time"])

## Step 1 - 建立多變量 window features

In [ ]:
model_features = [
    "traffic_total", "ucast_total", "avg_packet_size", "error_rate",
    "discard_rate", "broadcast_ratio", "multicast_ratio", "unknown_total",
]

window_rows = []
for port_id, g in features.groupby("port_id", sort=False):
    g = g.sort_values("timestamp").set_index("timestamp")
    wf = g[["device_id", "port_id", "port_role", "event_label", "event_id"]].copy()
    generated = {}
    for col in model_features:
        roll = g[col].rolling("30min", min_periods=3)
        generated[f"{col}_mean_30m"] = roll.mean()
        generated[f"{col}_std_30m"] = roll.std()
        generated[f"{col}_max_30m"] = roll.max()
        generated[f"{col}_min_30m"] = roll.min()
        generated[f"{col}_p95_30m"] = roll.quantile(0.95)
        generated[f"{col}_slope_30m"] = g[col].diff(6)
    wf = pd.concat([wf, pd.DataFrame(generated, index=wf.index)], axis=1)
    window_rows.append(wf.reset_index())

windows = pd.concat(window_rows, ignore_index=True).fillna(0)
X_cols = [c for c in windows.columns if c.endswith(("mean_30m", "std_30m", "max_30m", "min_30m", "p95_30m", "slope_30m"))]
X = windows[X_cols].replace([np.inf, -np.inf], 0).fillna(0)
print(windows.shape, len(X_cols))
display(windows[["timestamp", "port_id", "event_label"] + X_cols[:6]].head())

## Step 2 - 訓練 Isolation Forest

非監督式模型不需要 anomaly label，但 contamination 會影響異常比例。

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
model = IsolationForest(n_estimators=250, contamination=0.035, random_state=42)
model.fit(X_scaled)
windows["ml_anomaly_score"] = -model.score_samples(X_scaled)
windows["ml_is_anomaly"] = model.predict(X_scaled) == -1
windows["ml_method"] = "IsolationForest"

display(windows.loc[windows["ml_is_anomaly"], ["timestamp", "port_id", "event_label", "ml_anomaly_score"]].head(12))

## Step 3 - 視覺化：anomaly score trend

In [ ]:
port_id = "port-id7428"
p = windows[windows["port_id"] == port_id].copy()
threshold = windows["ml_anomaly_score"].quantile(0.965)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(p["timestamp"], p["ml_anomaly_score"], label="ML anomaly score", linewidth=1)
ax.axhline(threshold, color="tab:red", linestyle="--", label="global 96.5% threshold")
v = p[p["ml_is_anomaly"]]
ax.scatter(v["timestamp"], v["ml_anomaly_score"], color="tab:red", s=18, label="ML anomaly")
for _, event in event_catalog.iterrows():
    if event["port_id"] in [port_id, "MULTI"]:
        ax.axvspan(event["start_time"], event["end_time"], alpha=0.10, color="tab:red")
ax.set_title(f"{port_id} - Isolation Forest anomaly score")
ax.legend(loc="upper left")
ax.grid(alpha=0.25)
plt.tight_layout()
show_fig(fig)

## Step 4 - 視覺化：Top anomalous windows 與事件敏感度

In [ ]:
top_windows = windows.sort_values("ml_anomaly_score", ascending=False).head(15)
display(top_windows[["timestamp", "port_id", "event_label", "event_id", "ml_anomaly_score"]])

event_score = (
    windows.groupby("event_label")
    .agg(avg_score=("ml_anomaly_score", "mean"), p95_score=("ml_anomaly_score", lambda s: s.quantile(0.95)), anomaly_rate=("ml_is_anomaly", "mean"), rows=("timestamp", "size"))
    .sort_values("p95_score", ascending=False)
)
display(event_score)

fig, ax = plt.subplots(figsize=(12, 5))
event_score["anomaly_rate"].sort_values().plot(kind="barh", ax=ax, color="tab:blue")
ax.set_title("ML anomaly rate by simulated event")
ax.set_xlabel("anomaly rate")
plt.tight_layout()
show_fig(fig)

## Step 5 - 簡易 feature contribution

Isolation Forest 本身不是可解釋模型，這裡用 top anomalous windows 與全體 median 的 robust distance 作為教學用 contribution proxy。

In [ ]:
top = windows[windows["ml_is_anomaly"]].copy()
median = X.median()
mad = (X - median).abs().median().replace(0, 1)
contrib = ((top[X_cols] - median).abs() / mad).mean().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
contrib.sort_values().plot(kind="barh", ax=ax, color="tab:purple")
ax.set_title("Proxy feature contribution among anomalous windows")
ax.set_xlabel("mean robust distance")
plt.tight_layout()
show_fig(fig)

In [ ]:
output_cols = ["timestamp", "device_id", "port_id", "port_role", "event_label", "event_id", "ml_method", "ml_anomaly_score", "ml_is_anomaly"] + X_cols
out_path = DATA_PROCESSED / "ml_anomaly_scores.csv"
windows[output_cols].to_csv(out_path, index=False)
print(f"saved: {out_path}")